In [1]:
import requests
import pandas as pd
import os

JSON_URL = "https://remoteok.com/remote-jobs.json"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://remoteok.com/"
}

RAW_OUTPUT_PATH = "../data/raw/remoteok_raw.csv"


def fetch_remoteok_jobs():
    print("Fetching jobs from JSON feed...")

    response = requests.get(JSON_URL, headers=HEADERS)
    response.raise_for_status()

    data = response.json()

    # first element is metadata → skip
    jobs = data[1:]

    print(f"Total jobs fetched: {len(jobs)}")
    return jobs


def build_raw_dataframe(jobs):

    records = []

    for job in jobs:
        tags = job.get("tags") or []

        job_url = job.get("url")
        if job_url and job_url.startswith("/"):
            job_url = "https://remoteok.com" + job_url

        records.append({
            "Job Title": job.get("position"),
            "Company Name": job.get("company"),
            "Job Tags / Skills": ", ".join(tags) if tags else "N/A",
            "Location": job.get("location") or "Worldwide",
            "Job Type (Raw)": job.get("type"),
            "Epoch": job.get("epoch"),
            "Job URL": job_url
        })

    df = pd.DataFrame(records)
    print("Raw dataframe created.")
    return df


def save_raw_dataset(df):
    os.makedirs("../data/raw", exist_ok=True)
    df.to_csv(RAW_OUTPUT_PATH, index=False)
    print(f"Saved raw dataset to: {RAW_OUTPUT_PATH}")


def main():
    jobs = fetch_remoteok_jobs()
    df_raw = build_raw_dataframe(jobs)
    save_raw_dataset(df_raw)


if __name__ == "__main__":
    main()


Fetching jobs from JSON feed...
Total jobs fetched: 100
Raw dataframe created.
Saved raw dataset to: ../data/raw/remoteok_raw.csv
